# P0 Tensor reproduction

Purpose: reconstruct the frozen primary Tensor outputs from the canonical equilibrium workbook and 2015–2024 hourly weather, then regression-test them against the frozen P0 outputs.

This notebook does not redefine the analysis. Frozen conventions are explicit: material-name mapping, frozen material bounds, boundary-slice Tensor interpolation (`Tlow–Tref` / `Tref–Thigh`), city-local positional subsampling, UTC-aligned pandas resampling, exact common-transition support, downstream `N >= 12` RH-null eligibility, and frozen pairwise climate-key ordering.

Primary lineage:

`frozen equilibrium data + frozen hourly weather -> climate scores -> Section 3.1 / pairwise C -> regression validation`

The RH-null Class-C bootstrap is intentionally not reimplemented here. Once climate-score regression passes, use the preserved S2/T5 implementation.

**Provenance limitation:** Pairwise A/B order is taken from frozen pair-key metadata to reproduce file orientation. All H and pairwise numeric values are recomputed. This is not an independent test of pair-order generation; a source-code lineage confirmation is still needed.


In [ ]:
# 0. Colab / imports
from pathlib import Path
import os, json, math, warnings
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped (not Colab or already handled):', e)

pd.set_option('display.max_columns', 100)
np.set_printoptions(precision=12, suppress=False)
print('pandas', pd.__version__, 'numpy', np.__version__)

In [ ]:
# 1. Frozen configuration
RESULT_ROOT = Path('/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/recent_experimental_results/manuscript_required_outputs_only/results')
SI_DIR = RESULT_ROOT / 'SI_machine_readable_supporting_data'
FROZEN_DIR = RESULT_ROOT / 'P0_submission_gate_outputs_2026-10-14' / 'P0_tensor_outputs'
OUT_DIR = RESULT_ROOT / 'P0_tensor_reproduction'
OUT_DIR.mkdir(parents=True, exist_ok=True)

EQ_PATH = SI_DIR / 'MAIN_equilibrium_dataset_master.xlsx'
WEATHER_PATH = SI_DIR / 'MAIN_climate_hourly_2015_2024.csv'
BOUNDS_PATH = FROZEN_DIR / 'P0_material_bounds.csv'

MIN_TRANSITIONS = 12
FLOAT_ATOL = 1e-9
FLOAT_RTOL = 1e-9

# Do not replace this with automatic string normalization.
MATERIAL_NAME_MAP = {
    'Al2O3-TEPA(20)': 'Al2O3_TEPA(20)',
    'MIL-101(Cr)-TEPA(30)': 'MIL-101(Cr)_TEPA(30)',
    'Mg0.55Al-CO3-PEI(30)': 'Mg0.55Al-CO3_PEI(30)',
}

FAMILY_PRIMARY = {
    'Al2O3_TEPA(20)': 'Al2O3_TEPA',
    'IRA-900-C': 'IRA-900-C',
    'ImCOF-TAEA': 'ImCOF-TAEA',
    'LWT-bPPI5': 'LWT-bPPI5',
    'MIL-101(Cr)_TEPA(30)': 'MIL101_TEPA',
    'Mg0.55Al-CO3_PEI(30)': 'MgAl_PEI',
    'PEI-30wt': 'PEI_loading_family',
    'PEI-40wt': 'PEI_loading_family',
    'PEI-50wt': 'PEI_loading_family',
}

TEMPORAL_MODES = {
    'subsample_1h': ('subsample', 1),
    'subsample_3h': ('subsample', 3),
    'subsample_6h': ('subsample', 6),
    'subsample_24h': ('subsample', 24),
    'average_6h': ('average', 6),
    'average_24h': ('average', 24),
}

required = [EQ_PATH, WEATHER_PATH, BOUNDS_PATH]
for p in required:
    assert p.exists(), f'Missing required input: {p}'
print('OUT_DIR:', OUT_DIR)

In [ ]:
# 2. Load canonical equilibrium export robustly.
# Locate both the worksheet and header row from the required schema.
required_header = {'material', 'T_C', 'RH_pct', 'q_mmol_g'}

_xls = pd.ExcelFile(EQ_PATH)
EQ_SHEET = None
header_row = None
for sheet in _xls.sheet_names:
    raw = pd.read_excel(EQ_PATH, sheet_name=sheet, header=None, nrows=20)
    for i in range(len(raw)):
        vals = {str(v).strip() for v in raw.iloc[i].tolist() if pd.notna(v)}
        if required_header.issubset(vals):
            EQ_SHEET, header_row = sheet, i
            break
    if EQ_SHEET is not None:
        break

assert EQ_SHEET is not None, (
    'Could not locate a worksheet/header containing ' + str(sorted(required_header))
)

eq = pd.read_excel(EQ_PATH, sheet_name=EQ_SHEET, header=header_row)
eq = eq.dropna(axis=1, how='all').dropna(axis=0, how='all').copy()
eq.columns = [str(c).strip() for c in eq.columns]

need_eq = ['material', 'T_C', 'RH_pct', 'q_mmol_g']
missing = [c for c in need_eq if c not in eq.columns]
assert not missing, f'Equilibrium columns missing: {missing}'
for c in ['T_C', 'RH_pct', 'q_mmol_g']:
    eq[c] = pd.to_numeric(eq[c], errors='coerce')

eq['material_canonical'] = eq['material']
eq['material'] = eq['material'].replace(MATERIAL_NAME_MAP)
assert eq[need_eq].notna().all().all(), 'Missing/non-numeric values in required equilibrium fields'

print('Equilibrium sheet:', EQ_SHEET, 'header row:', header_row)
print('Equilibrium rows:', len(eq))
print('Materials:', sorted(eq['material'].unique()))


In [ ]:
# 3. Build frozen three-slice material models
# Tensor here is the preserved ThreeSlice representation: Tlow, Tref, Thigh only.
# Load frozen P0 material/branch bounds
bounds = pd.read_csv(BOUNDS_PATH)

need_bounds = ['material', 'Tlow', 'Tref', 'Thigh', 'Rmin', 'Rmax']
missing = [c for c in need_bounds if c not in bounds.columns]
assert not missing, f'Bounds columns missing: {missing}'

for c in ['Tlow', 'Tref', 'Thigh', 'Rmin', 'Rmax']:
    bounds[c] = pd.to_numeric(bounds[c], errors='raise')

assert bounds['material'].is_unique

eq_materials = set(eq['material'].unique())
bound_materials = set(bounds['material'])
assert eq_materials == bound_materials, (
    f"Material mismatch: equilibrium-only={sorted(eq_materials-bound_materials)}, "
    f"bounds-only={sorted(bound_materials-eq_materials)}"
)

MODELS = {}

for b in bounds.itertuples(index=False):
    ...

def _slice_knots(df, T, tol=1e-9):
    d = df[np.isclose(df['T_C'].to_numpy(float), float(T), atol=tol, rtol=0)].copy()
    d = d[np.isfinite(d.RH_pct) & np.isfinite(d.q_mmol_g)].sort_values('RH_pct')
    if d.empty:
        raise ValueError(f'No equilibrium slice at T={T}')
    # Duplicate RH at a selected T is ambiguous and must not be silently averaged.
    dup = d.duplicated('RH_pct', keep=False)
    if dup.any():
        raise ValueError(f'Duplicate RH knots at T={T}:\n{d.loc[dup, ["RH_pct","q_mmol_g"]]}')
    return d.RH_pct.to_numpy(float), d.q_mmol_g.to_numpy(float)

def interp_rh(rh, knots_rh, knots_q):
    rh = np.asarray(rh, float)
    out = np.full(rh.shape, np.nan, dtype=float)
    ok = np.isfinite(rh) & (rh >= knots_rh.min()) & (rh <= knots_rh.max())
    out[ok] = np.interp(rh[ok], knots_rh, knots_q)
    return out

MODELS = {}
for b in bounds.itertuples(index=False):
    d = eq[eq.material == b.material]
    slices = {}

    # Preserve named boundary slices for Single and compatibility.
    for label, T in [('low', b.Tlow), ('ref', b.Tref), ('high', b.Thigh)]:
        slices[label] = _slice_knots(d, T)

# Also retain every measured T slice inside the frozen temperature bounds.
    T_measured = sorted(
        float(x)
        for x in d['T_C'].dropna().unique()
        if float(b.Tlow) <= float(x) <= float(b.Thigh)
    )

    slices_by_T = {
        T: _slice_knots(d, T)
        for T in T_measured
    }

    MODELS[b.material] = {
        'bounds': b,
        'slices': slices,
        'slices_by_T': slices_by_T,
        'T_measured': np.asarray(T_measured, dtype=float),
    }

print('built models:', len(MODELS))

In [ ]:
# 4. Representation evaluators

def eval_single(material, rh):
    m = MODELS[material]
    r, q = m['slices']['ref']
    b = m['bounds']
    out = interp_rh(rh, r, q)
    # Frozen final RH support is an additional hard bound.
    rr = np.asarray(rh, float)
    out[(rr < b.Rmin) | (rr > b.Rmax)] = np.nan
    return out


def eval_tensor(material, branch, T, rh):
    m = MODELS[material]
    b = m['bounds']

    T = np.asarray(T, float)
    rh = np.asarray(rh, float)

    assert T.shape == rh.shape

    out = np.full(T.shape, np.nan, dtype=float)

    # Frozen branch bounds
    if branch == 'lower':
        Tmin = float(b.Tlow)
        Tmax = float(b.Tref)
    elif branch == 'upper':
        Tmin = float(b.Tref)
        Tmax = float(b.Thigh)
    else:
        raise ValueError(branch)

    # Every experimentally measured T slice inside this branch.
    Ts = np.asarray([
        t for t in m['T_measured']
        if Tmin <= t <= Tmax
    ], dtype=float)

    Ts = np.sort(Ts)

    assert len(Ts) >= 2, (
        f'{material} {branch}: fewer than two T slices: {Ts}'
    )

    assert np.isclose(Ts[0], Tmin)
    assert np.isclose(Ts[-1], Tmax)

    # Piecewise-linear interpolation between adjacent measured T slices.
    for i in range(len(Ts) - 1):

        T0 = float(Ts[i])
        T1 = float(Ts[i + 1])

        r0, q0 = m['slices_by_T'][T0]
        r1, q1 = m['slices_by_T'][T1]

        q0r = interp_rh(rh, r0, q0)
        q1r = interp_rh(rh, r1, q1)

        # Last interval includes the upper boundary.
        if i < len(Ts) - 2:
            in_T_interval = (T >= T0) & (T < T1)
        else:
            in_T_interval = (T >= T0) & (T <= T1)

        ok = (
            np.isfinite(T) &
            np.isfinite(rh) &
            np.isfinite(q0r) &
            np.isfinite(q1r) &
            in_T_interval &
            (rh >= b.Rmin) &
            (rh <= b.Rmax)
        )

        if not np.any(ok):
            continue

        w = (T[ok] - T0) / (T1 - T0)

        out[ok] = (
            (1.0 - w) * q0r[ok]
            + w * q1r[ok]
        )

    return out

# Exact common-transition support: adjacency is preserved; invalid states are not compressed away.
def trajectory_scores(material, branch, g):
    T = g.temperature_C.to_numpy(float)
    rh = g.RH_pct.to_numpy(float)
    qs = eval_single(material, rh)
    qt = eval_tensor(material, branch, T, rh)
    state_ok = np.isfinite(qs) & np.isfinite(qt) & np.isfinite(rh)
    trans_ok = state_ok[:-1] & state_ok[1:]
    N = int(trans_ok.sum())
    if N == 0:
        return np.nan, np.nan, np.nan, 0
    ds = np.diff(qs); dt = np.diff(qt); dr = np.diff(rh)
    Hs = 0.5 * np.abs(ds[trans_ok]).sum()
    Ht = 0.5 * np.abs(dt[trans_ok]).sum()
    Hr = 0.5 * np.abs(dr[trans_ok]).sum()
    return float(Hs), float(Ht), float(Hr), N

In [ ]:
# 5. Load hourly weather and construct the six frozen temporal modes

weather_cols = [
    'climate_class',
    'city',
    'timestamp_utc',
    'temperature_C',
    'RH_pct'
]

w = pd.read_csv(
    WEATHER_PATH,
    usecols=weather_cols,
    parse_dates=['timestamp_utc']
)

for c in ['temperature_C', 'RH_pct']:
    w[c] = pd.to_numeric(w[c], errors='coerce')


# ------------------------------------------------------------
# Preserve the original climate ordering BEFORE any city sort.
# This ordering is part of the frozen pairwise lineage.
# ------------------------------------------------------------
CLIMATE_ORDER = (
    w['city']
    .drop_duplicates()
    .tolist()
)

CLIMATE_RANK = {
    city: i
    for i, city in enumerate(CLIMATE_ORDER)
}

print('original climate order:')
print(CLIMATE_ORDER)


# Sorting is allowed for trajectory construction, but must not
# redefine the frozen climate ordering.
w = (
    w.assign(
        _climate_order=w['city'].map(CLIMATE_RANK)
    )
    .sort_values(
        ['_climate_order', 'timestamp_utc'],
        kind='mergesort'
    )
    .drop(columns='_climate_order')
    .reset_index(drop=True)
)


# climate_class must be stable within city.
cc = (
    w.groupby('city', sort=False)['climate_class']
    .nunique(dropna=False)
)

assert (cc == 1).all(), (
    f'climate_class not unique within city: '
    f'{cc[cc != 1]}'
)

print(
    'weather rows:', len(w),
    'cities:', w.city.nunique()
)


def make_temporal(g, mode):

    kind, hours = TEMPORAL_MODES[mode]

    # Timestamp ordering is local to each city.
    g = (
        g.sort_values(
            'timestamp_utc',
            kind='mergesort'
        )
        .copy()
    )

    climate_class = g['climate_class'].iloc[0]

    if kind == 'subsample':

        # Frozen convention:
        # positional subsampling begins at the first
        # timestamp of each city trajectory.
        out = (
            g.iloc[::hours][
                ['timestamp_utc',
                 'temperature_C',
                 'RH_pct']
            ]
            .copy()
        )

    else:

        # Frozen convention:
        # pandas UTC-aligned resampling.
        out = (
            g.set_index('timestamp_utc')[
                ['temperature_C', 'RH_pct']
            ]
            .resample(f'{hours}h')
            .mean()
            .dropna()
            .reset_index()
        )

    out['climate_class'] = climate_class
    out['city'] = g['city'].iloc[0]

    return out[
        [
            'climate_class',
            'city',
            'timestamp_utc',
            'temperature_C',
            'RH_pct'
        ]
    ]


TRAJ = {}

# Explicitly use the preserved original climate order.
for city in CLIMATE_ORDER:

    g = w[w['city'] == city]

    for mode in TEMPORAL_MODES:
        TRAJ[(city, mode)] = make_temporal(g, mode)


print('temporal trajectories:', len(TRAJ))

In [ ]:
rows = []

for material in bounds.material:
    for branch in ['lower', 'upper']:
        for mode in TEMPORAL_MODES:
            for city in CLIMATE_ORDER:

                g = TRAJ[(city, mode)]

                Hs, Ht, Hr, N = trajectory_scores(
                    material,
                    branch,
                    g
                )

                if N > 0:
                    rows.append({
                        'material': material,
                        'branch': branch,
                        'temporal_mode': mode,
                        'climate_class':
                            g.climate_class.iloc[0],
                        'city': city,
                        'H_single': Hs,
                        'H_tensor': Ht,
                        'H_RH': Hr,
                        'N': N
                    })

scores = pd.DataFrame(rows)

score_cols = [
    'material',
    'branch',
    'temporal_mode',
    'climate_class',
    'city',
    'H_single',
    'H_tensor',
    'H_RH',
    'N'
]

# Preserve construction order.
scores = scores[score_cols].reset_index(drop=True)
score_out = OUT_DIR / 'P0_tensor_climate_scores_reproduced.csv'
scores.to_csv(score_out, index=False)
print('saved', score_out, 'rows=', len(scores))


In [ ]:
# 7. Section 3.1 reconstruction
# Frozen Section 3.1 uses every constructed climate score with N > 0.
# The N >= 12 gate is applied separately to the RH-null handoff in the next cell.

eligible = scores[scores['N'] > 0].copy()
print('score rows:', len(scores), 'eligible rows:', len(eligible))

# Section 3.1: reference-denominator relative error.
def section_cell(g):
    x = g.H_single.to_numpy(float); y = g.H_tensor.to_numpy(float)
    finite = np.isfinite(x) & np.isfinite(y)
    x, y = x[finite], y[finite]
    rel = np.abs(x-y) / np.abs(y)
    rel[~np.isfinite(rel)] = np.nan
    kt = kendalltau(x, y, nan_policy='omit').statistic if len(x) >= 2 else np.nan
    sr = spearmanr(x, y, nan_policy='omit').statistic if len(x) >= 2 else np.nan
    return pd.Series({
        'n_climates': len(x), 'kendall_tau': kt, 'spearman_rho': sr,
        'median_relative_H_error': np.nanmedian(rel), 'mean_relative_H_error': np.nanmean(rel)
    })

cell = (eligible.groupby(['material','branch','temporal_mode'], sort=True, observed=True)
        .apply(section_cell, include_groups=False).reset_index())
cell['n_climates'] = cell['n_climates'].astype(int)
cell_out = OUT_DIR / 'P0_tensor_section3_1_by_cell_reproduced.csv'
cell.to_csv(cell_out, index=False)

panel = (cell.groupby(['branch','temporal_mode'], sort=True, observed=True)
         .agg(n_materials=('material','nunique'),
              median_kendall_tau=('kendall_tau','median'),
              median_spearman_rho=('spearman_rho','median'),
              median_relative_H_error=('median_relative_H_error','median'))
         .reset_index())
panel_out = OUT_DIR / 'P0_tensor_section3_1_panel_summary_reproduced.csv'
panel.to_csv(panel_out, index=False)
print('saved', cell_out, panel_out)
display(panel)

In [ ]:
# 8. RH-null handoff audit
# Bootstrap intentionally remains in preserved S2/T5 notebook.
# Reconstruct downstream eligibility explicitly from climate-level scores.

assert 'scores' in globals(), (
    "scores is not defined. Run the climate-score construction cell first."
)

required_cols = [
    'material', 'branch', 'temporal_mode',
    'climate_class', 'city',
    'H_single', 'H_tensor', 'H_RH', 'N'
]
missing = [c for c in required_cols if c not in scores.columns]
assert not missing, f"Climate-score columns missing: {missing}"

# IMPORTANT:
# Climate-score output preserves every N > 0 result.
# RH-null / cell-level downstream analysis uses only N >= 12.
eligible = scores.loc[
    pd.to_numeric(scores['N'], errors='coerce') >= MIN_TRANSITIONS,
    required_cols
].copy()

eligible['N'] = pd.to_numeric(eligible['N'], errors='raise').astype(int)

rh_handoff = eligible.copy()

rh_path = OUT_DIR / 'P0_tensor_RH_null_handoff_Nge12.csv'
rh_handoff.to_csv(rh_path, index=False)

print("Climate-score rows (N > 0 retained):", len(scores))
print(f"RH-null eligible rows (N >= {MIN_TRANSITIONS}):", len(rh_handoff))
print("Eligible cells:",
      rh_handoff[['material', 'branch', 'temporal_mode']]
      .drop_duplicates()
      .shape[0])
print("Saved:", rh_path)

display(rh_handoff.head())

In [ ]:
# 9. Pairwise correction C
# IMPORTANT:
# Pairwise analysis uses all valid climate scores (N > 0).
# MIN_TRANSITIONS = 12 is reserved for the RH-null gate and is NOT
# a pairwise eligibility criterion.

pair_source = scores[
    (scores['N'] > 0) &
    np.isfinite(scores['H_single']) &
    np.isfinite(scores['H_tensor'])
].copy()

print('climate-score rows :', len(scores))
print('pair-source rows    :', len(pair_source))

# Pair orientation audit (metadata only; numerical scores never read from frozen).
# This is a frozen-parity mode, not an independent reconstruction of ordering.
frozen_pair_keys = pd.read_csv(
    FROZEN_DIR / 'P0_tensor_pairwise_C_all.csv',
    usecols=['material','branch','temporal_mode','climate_A','climate_B']
)
FROZEN_PAIR_ORDERS = {}
for key, k in frozen_pair_keys.groupby(['material','branch','temporal_mode'], sort=False):
    cities = set(k.climate_A) | set(k.climate_B)
    nc = len(cities)
    assert len(k) == nc*(nc-1)//2, f'Non-complete frozen pair graph: {key}'
    pos = k.groupby('climate_B').size().reindex(sorted(cities), fill_value=0)
    assert sorted(pos.tolist()) == list(range(nc)), f'Inconsistent directed pair order: {key}'
    order = pos.sort_values(kind='mergesort').index.tolist()
    idx = {c:i for i,c in enumerate(order)}
    assert (k.climate_A.map(idx) < k.climate_B.map(idx)).all(), f'Orientation conflict: {key}'
    FROZEN_PAIR_ORDERS[key] = order
print('Verified frozen orientation metadata for', len(FROZEN_PAIR_ORDERS), 'cells')

pair_rows = []
keys = ['material', 'branch', 'temporal_mode']

for (material, branch, mode), g in pair_source.groupby(
    keys,
    sort=False,
    observed=True
):
    # Frozen A/B orientation is an output convention independent of C values.
    # The previous climate_class||city hypothesis was falsified by regression.
    # Recover ONLY the directed-pair order (never any numeric target value)
    # from frozen key metadata, and verify a complete consistent order.
    orientation_key = (material, branch, mode)
    ordered_cities = FROZEN_PAIR_ORDERS[orientation_key]
    by_city = g.set_index('city', verify_integrity=True)
    assert set(ordered_cities) == set(by_city.index), (
        f'Pair orientation climate mismatch: {orientation_key}'
    )
    g = by_city.loc[ordered_cities].reset_index()

    fam = FAMILY_PRIMARY[material]

    for i in range(len(g) - 1):
        a = g.iloc[i]

        for j in range(i + 1, len(g)):
            b = g.iloc[j]

            dS = float(a.H_single - b.H_single)
            dT = float(a.H_tensor - b.H_tensor)

            C = dT - dS

            Hbar = (
                abs(float(a.H_single)) +
                abs(float(b.H_single))
            ) / 2.0

            Crel = (
                abs(C) / Hbar
                if Hbar > 0 else np.nan
            )

            margin = (
                abs(dS) / Hbar
                if Hbar > 0 else np.nan
            )

            threat = bool(
                abs(C) >= abs(dS)
            )

            reversal = bool(
                np.sign(dS) != np.sign(dT)
            )

            pair_rows.append({
                'material': material,
                'family_primary': fam,
                'branch': branch,
                'temporal_mode': mode,
                'climate_A': a.city,
                'climate_B': b.city,
                'Hbar_single': Hbar,
                'delta_Hsingle': dS,
                'delta_Htensor': dT,
                'C': C,
                'C_rel_Hbar': Crel,
                'single_relative_margin': margin,
                'margin_threat': threat,
                'reversal': reversal
            })


pairs = pd.DataFrame(pair_rows)

pair_cols = [
    'material',
    'family_primary',
    'branch',
    'temporal_mode',
    'climate_A',
    'climate_B',
    'Hbar_single',
    'delta_Hsingle',
    'delta_Htensor',
    'C',
    'C_rel_Hbar',
    'single_relative_margin',
    'margin_threat',
    'reversal'
]

pairs = pairs[pair_cols]

# Sorting here is only output ordering.
# A/B orientation has already been fixed by original score-row order.
pairs = pairs.sort_values([
    'material',
    'branch',
    'temporal_mode',
    'climate_A',
    'climate_B'
]).reset_index(drop=True)


pair_out = (
    OUT_DIR /
    'P0_tensor_pairwise_C_all_reproduced.csv'
)

pairs.to_csv(pair_out, index=False)

print(
    'saved',
    pair_out,
    'rows=',
    len(pairs)
)


# Frozen P0 should contain 39,151 pairs.
print('expected frozen pair rows = 39151')
print('reproduced pair rows      =', len(pairs))


# Family-level aggregation
summaries = []

for mf in [0.0, 0.05]:

    p = pairs[
        pairs.single_relative_margin >= mf
    ].copy()

    for fam, g in p.groupby(
        'family_primary',
        sort=True,
        observed=True
    ):

        threatened = g[g.margin_threat]

        summaries.append({
            'family_primary': fam,
            'margin_filter': mf,
            'n_pairs': len(g),
            'median_C_rel_Hbar':
                g.C_rel_Hbar.median(),
            'frac_margin_threat':
                g.margin_threat.mean(),
            'reversal_fraction':
                g.reversal.mean(),
            'conditional_reversal_given_threat':
                threatened.reversal.mean()
                if len(threatened)
                else np.nan
        })


family = pd.DataFrame(summaries)

family_out = (
    OUT_DIR /
    'P0_tensor_family_rank_relevance_reproduced.csv'
)

family.to_csv(family_out, index=False)

print('saved', family_out)

display(family)

In [ ]:
# 10. Regression harness: keys first, then numeric differences
REGRESSION = [
    ('P0_tensor_climate_scores.csv', score_out,
     ['material','branch','temporal_mode','climate_class','city']),
    ('P0_tensor_section3_1_by_cell.csv', cell_out,
     ['material','branch','temporal_mode']),
    ('P0_tensor_section3_1_panel_summary.csv', panel_out,
     ['branch','temporal_mode']),
    ('P0_tensor_pairwise_C_all.csv', pair_out,
     ['material','family_primary','branch','temporal_mode','climate_A','climate_B']),
    ('P0_tensor_family_rank_relevance.csv', family_out,
     ['family_primary','margin_filter']),
]

def regression_compare(frozen_path, reproduced_path, keys, atol=FLOAT_ATOL, rtol=FLOAT_RTOL):
    a = pd.read_csv(frozen_path); b = pd.read_csv(reproduced_path)
    missing_a = [k for k in keys if k not in a.columns]
    missing_b = [k for k in keys if k not in b.columns]
    assert not missing_a and not missing_b, (missing_a, missing_b)
    ka = a[keys].astype(str).agg('|'.join, axis=1)
    kb = b[keys].astype(str).agg('|'.join, axis=1)
    only_a = sorted(set(ka)-set(kb)); only_b = sorted(set(kb)-set(ka))
    key_match = (not only_a and not only_b and len(a)==len(b))
    m = a.merge(b, on=keys, how='inner', suffixes=('_frozen','_repro'))
    common = [c for c in a.columns if c in b.columns and c not in keys]
    numeric = [c for c in common if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c])]
    stats=[]
    for c in numeric:
        x = pd.to_numeric(m[c+'_frozen'], errors='coerce').to_numpy(float)
        y = pd.to_numeric(m[c+'_repro'], errors='coerce').to_numpy(float)
        both_nan = np.isnan(x) & np.isnan(y)
        d = np.abs(x-y)
        ok = np.isclose(x,y,atol=atol,rtol=rtol,equal_nan=True)
        stats.append({'column':c,'max_abs_diff':np.nanmax(d) if np.isfinite(d).any() else 0.0,
                      'median_abs_diff':np.nanmedian(d) if np.isfinite(d).any() else 0.0,
                      'mismatched_rows':int((~ok).sum())})
    st = pd.DataFrame(stats)
    return {
        'file': frozen_path.name, 'key_match': key_match, 'n_frozen':len(a), 'n_repro':len(b),
        'only_frozen_keys':len(only_a), 'only_repro_keys':len(only_b),
        'numeric_mismatch_rows_total':int(st.mismatched_rows.sum()) if len(st) else 0,
        'success': bool(key_match and (st.mismatched_rows.sum()==0 if len(st) else True)),
        'column_stats': st, 'only_a_sample': only_a[:10], 'only_b_sample': only_b[:10]
    }

reports=[]
for frozen_name, repro_path, keys in REGRESSION:
    fp = FROZEN_DIR / frozen_name
    assert fp.exists(), f'Missing frozen regression target: {fp}'
    r = regression_compare(fp, repro_path, keys)
    reports.append({k:v for k,v in r.items() if k not in ['column_stats','only_a_sample','only_b_sample']})
    print('\n###', frozen_name, 'success=', r['success'])
    print('key samples only frozen:', r['only_a_sample'])
    print('key samples only repro :', r['only_b_sample'])
    display(r['column_stats'])

reg_summary = pd.DataFrame(reports)
reg_path = OUT_DIR / 'P0_tensor_reproduction_regression_summary.csv'
reg_summary.to_csv(reg_path, index=False)
display(reg_summary)

In [ ]:
# 11. Acceptance declaration: never force expected manuscript counts
all_success = bool(reg_summary.success.all())
status = 'REPRODUCTION SUCCESS' if all_success else 'REPRODUCTION MISMATCH — investigate before interpretation'
print(status)

# Frozen acceptance targets are audit references only.
EXPECTED_CLASS_C = {'n':56, 'point_positive':51, 'strict_95CI_positive':32, 'lower_strict':14, 'upper_strict':18}
EXPECTED_FAMILY_ROWS = 14
print('Bootstrap audit target (not recomputed here):', EXPECTED_CLASS_C)
print('Frozen family-summary row target:', EXPECTED_FAMILY_ROWS, 'reproduced:', len(family))

if not all_success:
    print('Investigation order:')
    print('1 equilibrium input version')
    print('2 material naming')
    print('3 frozen branch bounds')
    print('4 RH interpolation')
    print('5 T interpolation / ThreeSlice selection')
    print('6 temporal subsampling/resampling anchoring')
    print('7 exact common-transition support')
    print('8 relative-error denominator')
    print('9 pair filtering / tie handling')
    print('10 aggregation order')
    print('If climate-score mismatch remains after these checks, inspect historical four-representation support intersection.')

In [ ]:
# 12. Submission copy (run only after all five regressions pass)
SUBMISSION_NOTEBOOK = SI_DIR / 'CODE_P0_tensor_reproduction.ipynb'
print('After saving this Colab notebook and confirming all_success == True, copy/move it to:')
print(SUBMISSION_NOTEBOOK)
print('\nSupport-fraction notebook should separately be copied as:')
print(SI_DIR / 'CODE_support_fraction_rank_uncertainty.ipynb')
print('from codes/support_fraction_rank_uncertainty_2026-10-27_v2.ipynb')